# Credit Card Fraud: EDA and Preprocessing

## Data Loading

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')
np.set_printoptions(suppress=True, precision=4)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
PROJECT_ROOT = Path.cwd()
RAW_PATH = PROJECT_ROOT / 'Data' / 'raw' / 'fraud_train.csv'
PROCESSED_PATH = PROJECT_ROOT / 'Data' / 'processed' / 'fraud_train_preprocessed.csv'

FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Training data not found: {RAW_PATH}")

RAW_PATH

When loading the files, python will automatically define which datatypes they will use, however, since card number(`cc_num`), zip code (`zip`) and the transaction number (`trans_num`) will be identified as number, we define them as string beforehand.

In [ ]:
identifier_types = {
    'cc_num': 'string',
    'zip': 'string',
    'trans_num': 'string',
}

# low_memory is used to make sure interpreter read through the whole dataset before making guesses of the data types
# Load one working DataFrame to avoid duplicating this 1.3-million-row dataset in memory.
df = pd.read_csv(RAW_PATH, dtype=identifier_types, low_memory=False)

print(f'Rows: {len(df):,}')
print(f'Columns: {df.shape[1]}')
display(df.head())

## Data Cleaning & Standardization

In [ ]:
quality_summary = pd.Series({
    'rows': len(df),
    'columns': df.shape[1],
    'missing_cells': int(df.isna().sum().sum()),
    'duplicate_rows': int(df.duplicated().sum()),
    'duplicate_transaction_numbers': int(df['trans_num'].duplicated().sum()),
    'fraud_cases': int(df['is_fraud'].sum()),
    'fraud_rate': df['is_fraud'].mean(),
})
display(quality_summary.to_frame('value'))
display(df.dtypes.to_frame('dtype'))

In [ ]:
# Removing `Unnamed:` since it does not have any meaning
export_index_columns = [c for c in df.columns if c.startswith('Unnamed:')]
df = df.drop(columns=export_index_columns)
print('Removed:', export_index_columns)

In [ ]:
# reorganizing dates
df['trans_date_trans_time'] = pd.to_datetime(
    df['trans_date_trans_time'], format='mixed', dayfirst=True, errors='coerce'
)
df['dob'] = pd.to_datetime(
    df['dob'], format='mixed', dayfirst=True, errors='coerce'
)

date_check = pd.Series({
    'invalid_transaction_dates': int(df['trans_date_trans_time'].isna().sum()),
    'invalid_birth_dates': int(df['dob'].isna().sum()),
    'first_transaction': df['trans_date_trans_time'].min(),
    'last_transaction': df['trans_date_trans_time'].max(),
})
display(date_check.to_frame('value'))

### Card Number and Customer Profile Matching
Building customer profile using `first`, `last`, `dob`, `street` and check the total number with the total number of card number, to make sure there is a 1-1 relationship for later customer-level analysis

In [ ]:
customer_profiles = ['first', 'last', 'dob', 'street']
profiles_per_card = (
    df.groupby('cc_num', dropna=False)[customer_profiles]
      .apply(lambda x: len(x.drop_duplicates()))
)

card_identifier_summary = pd.Series({
    'scientific_notation_rows': int(df['cc_num'].str.contains('[eE]', regex=True, na=False).sum()),
    'unique_card_strings': df['cc_num'].nunique(),
    'unique_customer_profiles': df[customer_profiles].drop_duplicates().shape[0],
    'card_strings_linked_to_multiple_profiles': int(profiles_per_card.gt(1).sum()),
})
display(card_identifier_summary.to_frame('value'))

assert card_identifier_summary['scientific_notation_rows'] == 0, (
    'Training card identifiers unexpectedly contain scientific notation.'
)

### Categorical Columns
Removing prefix for `merchant`, along with making sure there is no error for `category`, `gender`, and `zip` when the user type in the information

In [ ]:
df['merchant'] = df['merchant'].str.removeprefix('fraud_').str.strip().astype('string')
df['category'] = df['category'].str.strip().str.lower().astype('string')
df['gender'] = df['gender'].str.strip().str.upper().astype('string')
df['zip'] = df['zip'].str.strip().str.replace(r'\.0$', '', regex=True).str.zfill(5)

for column in ['city', 'state', 'job']:
    df[column] = df[column].str.strip().astype('string')

# display(df[['merchant', 'category', 'gender', 'city', 'state', 'zip', 'job']].head())

In [ ]:
categorical_columns = ['merchant', 'category', 'gender', 'zip']

for column in categorical_columns:
    print(f'\n===== {column.upper()} =====')
    print(f'Total non-null values: {df[column].count():,}')
    print(f'Unique values: {df[column].nunique():,}')
    # display(df[column].value_counts().to_frame(name='count'))

### Cities, States and Zip Code
From the below result, we see that there are some city names happening in multiple states. With the zip code number > city name number, that means some city have different zip code. And also, different cities can have the same population

In [ ]:
pd.Series({
    'unique_city_names': df['city'].nunique(),
    'unique_city_state_pairs': df[['city', 'state']].drop_duplicates().shape[0],
    'unique_population_values': df['city_pop'].nunique(),
    'unique_zip_codes': df['zip'].nunique(),
}).to_frame('value')

### Unix Timestamp
Since the unix timestamp is off from the transaction timestamp with a constant amount, we will stick with the transaction time and drop this column

In [ ]:
unix_time_dt = pd.to_datetime(df['unix_time'], unit='s', errors='coerce')
unix_gap_days = (df['trans_date_trans_time'] - unix_time_dt).dt.total_seconds() / 86_400

pd.Series({
    'unix_date_min': unix_time_dt.min(),
    'unix_date_max': unix_time_dt.max(),
    'minimum_gap_days': unix_gap_days.min(),
    'maximum_gap_days': unix_gap_days.max(),
}).to_frame('value')

In [ ]:
df = df.drop(columns=['unix_time'], errors='ignore')

## Feature Engineering

### Transaction Time
The below deal with transaction time and split it into four six field, along with using sin and cos to make sure the time is cyclical and easier for the model to understand(for `trans_hour` and `trans_day_of_week`)

In [ ]:
transaction_time = df['trans_date_trans_time']

df['trans_hour'] = transaction_time.dt.hour.astype('int8')
df['trans_day_of_week'] = transaction_time.dt.dayofweek.astype('int8')
df['trans_month'] = transaction_time.dt.month.astype('int8')
df['is_weekend'] = transaction_time.dt.dayofweek.ge(5).astype('int8')
df['is_night'] = transaction_time.dt.hour.between(0, 5).astype('int8')
df['age_at_transaction'] = (
    (transaction_time - df['dob']).dt.days / 365.2425
).clip(lower=0).astype('float32')

# Cyclic encodings keep 23:00 close to 00:00 and Sunday close to Monday. This is using semi-circle coordinate by applying sin and cos function and using their coordinate
df['hour_sin'] = np.sin(2 * np.pi * df['trans_hour'] / 24).astype('float32')
df['hour_cos'] = np.cos(2 * np.pi * df['trans_hour'] / 24).astype('float32')
df['day_sin'] = np.sin(2 * np.pi * df['trans_day_of_week'] / 7).astype('float32')
df['day_cos'] = np.cos(2 * np.pi * df['trans_day_of_week'] / 7).astype('float32')

display(df[[
    'trans_hour', 'trans_day_of_week', 'trans_month', 'is_weekend',
    'is_night', 'age_at_transaction', 'hour_sin', 'hour_cos'
]].describe())

### EDA: Transaction Time

In [ ]:
from matplotlib.ticker import FuncFormatter

day_labels = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
count_formatter = FuncFormatter(lambda value, _: f'{value / 1_000:,.0f}K')

hour_frequency = df['trans_hour'].value_counts().reindex(range(24), fill_value=0)
weekday_frequency = df['trans_day_of_week'].value_counts().reindex(range(7), fill_value=0)

# Normalize month-of-year frequency by the number of observed dates in each month.
transaction_date = df['trans_date_trans_time'].dt.normalize()
daily_frequency = transaction_date.value_counts().sort_index()
monthly_daily_frequency = (
    daily_frequency.groupby(daily_frequency.index.month).mean()
    .reindex(range(1, 13))
)

weekend_frequency = df['is_weekend'].value_counts().reindex([0, 1], fill_value=0)
night_frequency = df['is_night'].value_counts().reindex([0, 1], fill_value=0)

age_bins = np.arange(10, 111, 10)
age_groups = pd.cut(
    df['age_at_transaction'], bins=age_bins, right=False,
    labels=[f'{start}–{start + 9}' for start in age_bins[:-1]],
)
age_frequency = age_groups.value_counts(sort=False)

fig, axes = plt.subplots(2, 3, figsize=(17, 10), constrained_layout=True)

axes[0, 0].bar(hour_frequency.index, hour_frequency.values, color='#4C78A8')
axes[0, 0].set_title('Transaction Frequency by Hour')
axes[0, 0].set_xlabel('Hour of day')
axes[0, 0].set_ylabel('Transactions')
axes[0, 0].set_xticks(range(0, 24, 2))
axes[0, 0].yaxis.set_major_formatter(count_formatter)

axes[0, 1].bar(day_labels, weekday_frequency.values, color='#59A14F')
axes[0, 1].set_title('Transaction Frequency by Day of Week')
axes[0, 1].set_xlabel('Day')
axes[0, 1].set_ylabel('Transactions')
axes[0, 1].tick_params(axis='x', rotation=35)
axes[0, 1].yaxis.set_major_formatter(count_formatter)

axes[0, 2].bar(month_labels, monthly_daily_frequency.values, color='#F28E2B')
axes[0, 2].set_title('Average Daily Transaction Frequency by Month')
axes[0, 2].set_xlabel('Calendar month')
axes[0, 2].set_ylabel('Average transactions per observed day')

weekend_percent = weekend_frequency.div(weekend_frequency.sum()).mul(100)
axes[1, 0].bar(['Weekday', 'Weekend'], weekend_percent.values, color=['#4C78A8', '#E15759'])
axes[1, 0].set_title('Share of Transactions: Weekday vs Weekend')
axes[1, 0].set_ylabel('Transactions (%)')
for position, value in enumerate(weekend_percent):
    axes[1, 0].text(position, value, f'{value:.1f}%', ha='center', va='bottom')

night_percent = night_frequency.div(night_frequency.sum()).mul(100)
axes[1, 1].bar(['06:00–23:59', '00:00–05:59'], night_percent.values,
               color=['#76B7B2', '#B07AA1'])
axes[1, 1].set_title('Share of Transactions: Day/Evening vs Night')
axes[1, 1].set_ylabel('Transactions (%)')
for position, value in enumerate(night_percent):
    axes[1, 1].text(position, value, f'{value:.1f}%', ha='center', va='bottom')

axes[1, 2].bar(age_frequency.index.astype(str), age_frequency.values, color='#9C755F')
axes[1, 2].set_title('Transaction Frequency by Customer Age')
axes[1, 2].set_xlabel('Age at transaction')
axes[1, 2].set_ylabel('Transactions')
axes[1, 2].tick_params(axis='x', rotation=35)
axes[1, 2].yaxis.set_major_formatter(count_formatter)

fig.suptitle('Preliminary Customer Spending-Frequency Patterns', fontsize=16)
plt.savefig(
    FIGURE_DIR / "figure-1.pdf",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
frequency_summary = pd.Series({
    'busiest_hour': int(hour_frequency.idxmax()),
    'busiest_weekday': day_labels[int(weekday_frequency.idxmax())],
    'highest_average_daily_month': month_labels[int(monthly_daily_frequency.idxmax()) - 1],
    'weekend_transaction_share': weekend_percent.loc[1] / 100,
    'night_transaction_share': night_percent.loc[1] / 100,
    'most_active_age_group': str(age_frequency.idxmax()),
})
print(frequency_summary)

### EDA: Transaction Time w/ Fraud Analysis
- **Fraud cases** measure fraud volume. A busy period may contain many fraud cases simply because it contains many transactions. **Fraud rate** measures risk: the percentage of transactions in that group that are fraudulent.

In [ ]:
def summarize_fraud_by(group):
    return (
        df.assign(_group=group)
          .dropna(subset=['_group'])
          .groupby('_group', observed=True)['is_fraud']
          .agg(transactions='size', fraud_cases='sum', fraud_rate='mean')
    )


hour_fraud = summarize_fraud_by(df['trans_hour']).reindex(range(24), fill_value=0)
weekday_fraud = summarize_fraud_by(df['trans_day_of_week']).reindex(range(7), fill_value=0)
month_fraud = summarize_fraud_by(df['trans_month']).reindex(range(1, 13), fill_value=0)
weekend_fraud = summarize_fraud_by(df['is_weekend']).reindex([0, 1], fill_value=0)
night_fraud = summarize_fraud_by(df['is_night']).reindex([0, 1], fill_value=0)
age_fraud = summarize_fraud_by(age_groups)


def plot_fraud_rate_and_cases(ax, summary, labels, title, rotation=0):
    positions = np.arange(len(summary))
    fraud_rate_percent = summary['fraud_rate'].mul(100)

    bars = ax.bar(positions, fraud_rate_percent, color='#E15759', alpha=0.82)
    ax.set_title(title)
    ax.set_ylabel('Fraud rate (%)', color='#B23A3A')
    ax.tick_params(axis='y', labelcolor='#B23A3A')
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=rotation)

    cases_axis = ax.twinx()
    cases_axis.plot(
        positions, summary['fraud_cases'], color='#1F4E79',
        marker='o', linewidth=1.8, markersize=4,
    )
    cases_axis.set_ylabel('Fraud cases', color='#1F4E79')
    cases_axis.tick_params(axis='y', labelcolor='#1F4E79')

    # One compact legend for the two different measurements.
    rate_proxy = plt.Line2D([0], [0], color='#E15759', linewidth=8, alpha=0.82)
    cases_proxy = plt.Line2D([0], [0], color='#1F4E79', marker='o')
    ax.legend(
        [rate_proxy, cases_proxy], ['Fraud rate', 'Fraud cases'],
        loc='upper left', fontsize=8,
    )


fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)

plot_fraud_rate_and_cases(
    axes[0, 0], hour_fraud, [str(hour) for hour in range(24)],
    'Fraud Risk and Volume by Hour', rotation=0,
)
axes[0, 0].set_xlabel('Hour of day')

plot_fraud_rate_and_cases(
    axes[0, 1], weekday_fraud, day_labels,
    'Fraud Risk and Volume by Weekday', rotation=35,
)

plot_fraud_rate_and_cases(
    axes[0, 2], month_fraud, month_labels,
    'Fraud Risk and Volume by Calendar Month', rotation=0,
)

plot_fraud_rate_and_cases(
    axes[1, 0], weekend_fraud, ['Weekday', 'Weekend'],
    'Fraud Risk and Volume: Weekday vs Weekend', rotation=0,
)

plot_fraud_rate_and_cases(
    axes[1, 1], night_fraud, ['06:00–23:59', '00:00–05:59'],
    'Fraud Risk and Volume: Day/Evening vs Night', rotation=0,
)

plot_fraud_rate_and_cases(
    axes[1, 2], age_fraud, age_fraud.index.astype(str),
    'Fraud Risk and Volume by Customer Age', rotation=35,
)

fig.suptitle('Fraud Rate and Fraud-Case Volume Across Spending Patterns', fontsize=16)
plt.savefig(
    FIGURE_DIR / "figure-2.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

fraud_timing_summary = pd.Series({
    'hour_with_most_fraud_cases': int(hour_fraud['fraud_cases'].idxmax()),
    'hour_with_highest_fraud_rate': int(hour_fraud['fraud_rate'].idxmax()),
    'highest_hourly_fraud_rate': hour_fraud['fraud_rate'].max(),
    'weekday_with_most_fraud_cases': day_labels[int(weekday_fraud['fraud_cases'].idxmax())],
    'weekday_with_highest_fraud_rate': day_labels[int(weekday_fraud['fraud_rate'].idxmax())],
    'month_with_most_fraud_cases': month_labels[int(month_fraud['fraud_cases'].idxmax()) - 1],
    'month_with_highest_fraud_rate': month_labels[int(month_fraud['fraud_rate'].idxmax()) - 1],
    'night_fraud_rate': night_fraud.loc[1, 'fraud_rate'],
    'non_night_fraud_rate': night_fraud.loc[0, 'fraud_rate'],
    'highest_risk_age_group': str(age_fraud['fraud_rate'].idxmax()),
})
display(fraud_timing_summary.to_frame('value'))

# Detailed hourly table makes the highest-volume and highest-rate hours auditable.
display(
    hour_fraud.assign(fraud_rate_percent=hour_fraud['fraud_rate'].mul(100))
              .sort_values('fraud_rate', ascending=False)
              .head(10)
)

### Merchant and Customer Locations
The below is trying to understand whether geographic location matter and whether it can becomes predictors that the model can use

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized great-circle distance between two coordinate pairs."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1
    a = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2) ** 2
    )
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))


df['merchant_lat_delta'] = (df['merch_lat'] - df['lat']).astype('float32')
df['merchant_long_delta'] = (df['merch_long'] - df['long']).astype('float32')
df['merchant_distance_km'] = haversine_km(
    df['lat'].to_numpy(), df['long'].to_numpy(),
    df['merch_lat'].to_numpy(), df['merch_long'].to_numpy(),
).astype('float32')

# Direction of merchant displacement; sine/cosine avoids a discontinuity at 360 degrees.
bearing_radians = np.arctan2(
    np.radians(df['merchant_long_delta']),
    np.radians(df['merchant_lat_delta']),
)
df['merchant_direction_sin'] = np.sin(bearing_radians).astype('float32')
df['merchant_direction_cos'] = np.cos(bearing_radians).astype('float32')

display(df[[
    'merchant_lat_delta', 'merchant_long_delta', 'merchant_distance_km',
    'merchant_direction_sin', 'merchant_direction_cos'
]].describe())

### EDA: Merchant and Customer Location

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

for ax, x, y, title in [
    (axes[0], 'long', 'lat', 'Customer Locations'),
    (axes[1], 'merch_long', 'merch_lat', 'Merchant Locations'),
]:
    plot = ax.hexbin(
        df[x], df[y], C=df['is_fraud'], reduce_C_function=np.mean,
        gridsize=45, mincnt=50, cmap='YlOrRd', vmin=0,
    )
    ax.set_title(f'{title}: Mean Fraud Rate by Map Cell')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

fig.colorbar(plot, ax=axes, label='Observed fraud rate')
plt.savefig(
    FIGURE_DIR / "figure-6.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

distance_bins = pd.qcut(df['merchant_distance_km'], q=10, duplicates='drop')
distance_rate = df.groupby(distance_bins, observed=True)['is_fraud'].agg(['mean', 'size'])
fig, ax = plt.subplots(figsize=(9, 4))
distance_rate['mean'].mul(100).plot(ax=ax, marker='o', color='#E45756')
ax.axhline(df['is_fraud'].mean() * 100, color='gray', linestyle='--', label='Overall rate')
ax.set_title('Fraud Rate Across Customer-to-Merchant Distance Deciles')
ax.set_xlabel('Merchant-distance decile')
ax.set_ylabel('Fraud rate (%)')
ax.tick_params(axis='x', rotation=35)
ax.legend()
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "figure-7.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

### Customer History
This is another set of feature generated from the transaction time, it help us to understand the total number of transaction for one customer, the minutes between each transaction, amount ratio compare to previous, and whether this is a new customer

In [ ]:
# Create a non-PII grouping key from the four profile fields, then sort chronologically.
df['customer_key'] = pd.util.hash_pandas_object(
    df[customer_profiles].astype('string'), index=False
).astype('uint64').astype('string')

df = df.sort_values(['customer_key', 'trans_date_trans_time'], kind='stable').copy()
customer_group = df.groupby('customer_key', sort=False)

df['customer_prior_txn_count'] = customer_group.cumcount().astype('int32')
df['minutes_since_previous_txn'] = (
    customer_group['trans_date_trans_time'].diff().dt.total_seconds().div(60)
).astype('float32')
df['is_first_observed_txn'] = df['minutes_since_previous_txn'].isna().astype('int8')

prior_amount_sum = customer_group['amt'].cumsum() - df['amt']
prior_amount_mean = prior_amount_sum.div(df['customer_prior_txn_count'].replace(0, np.nan))
df['amount_to_prior_mean'] = (df['amt'] / prior_amount_mean.replace(0, np.nan)).astype('float32')
df['no_prior_amount_history'] = prior_amount_mean.isna().astype('int8')

# Count whether this customer has used the merchant before; no target values are used.
df['customer_merchant_prior_count'] = (
    df.groupby(['customer_key', 'merchant'], sort=False).cumcount().astype('int32')
)
df['is_new_merchant_for_customer'] = df['customer_merchant_prior_count'].eq(0).astype('int8')

# Neutral fills plus explicit missing-history flags keep every modeling row usable.
df['minutes_since_previous_txn'] = df['minutes_since_previous_txn'].fillna(-1)
df['amount_to_prior_mean'] = df['amount_to_prior_mean'].replace([np.inf, -np.inf], np.nan).fillna(1)
df = df.sort_values('trans_date_trans_time', kind='stable').reset_index(drop=True)

display(df[[
    'customer_prior_txn_count', 'minutes_since_previous_txn', 'amount_to_prior_mean',
    'customer_merchant_prior_count', 'is_new_merchant_for_customer'
]].describe())

### EDA: Customer History Analysis

In [ ]:
def plot_clipped_histogram(ax, series, title, xlabel, bins=35, lower=None):
    values = series.dropna()
    if lower is not None:
        values = values[values.ge(lower)]
    upper = values.quantile(0.99)
    displayed = values.clip(upper=upper)
    ax.hist(displayed, bins=bins, color='#4C78A8', edgecolor='white', alpha=0.9)
    ax.axvline(values.median(), color='#E15759', linestyle='--',
               label=f'Median: {values.median():,.2f}')
    ax.set_title(f'{title}\n(display capped at 99th percentile: {upper:,.2f})')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Transactions')
    ax.yaxis.set_major_formatter(count_formatter)
    ax.legend(fontsize=8)


fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)

plot_clipped_histogram(
    axes[0, 0], df['customer_prior_txn_count'],
    'Previous Transactions Observed for Customer', 'Prior transaction count',
)
plot_clipped_histogram(
    axes[0, 1], df['minutes_since_previous_txn'],
    'Time Since Customer’s Previous Transaction', 'Minutes since previous transaction',
    lower=0,
)
plot_clipped_histogram(
    axes[0, 2], df.loc[df['no_prior_amount_history'].eq(0), 'amount_to_prior_mean'],
    'Current Amount Relative to Prior Average', 'Current amount / prior mean amount',
)
plot_clipped_histogram(
    axes[1, 0], df['customer_merchant_prior_count'],
    'Previous Transactions at the Same Merchant', 'Prior customer–merchant count',
)

first_share = df['is_first_observed_txn'].value_counts(normalize=True).reindex([0, 1]).mul(100)
axes[1, 1].bar(['Has history', 'First observed'], first_share.values,
               color=['#59A14F', '#F28E2B'])
axes[1, 1].set_title('Customer History Availability')
axes[1, 1].set_ylabel('Transactions (%)')
for position, value in enumerate(first_share):
    axes[1, 1].text(position, value, f'{value:.1f}%', ha='center', va='bottom')

merchant_share = df['is_new_merchant_for_customer'].value_counts(normalize=True).reindex([0, 1]).mul(100)
axes[1, 2].bar(['Previously used merchant', 'New merchant'], merchant_share.values,
               color=['#76B7B2', '#B07AA1'])
axes[1, 2].set_title('Customer Familiarity with Merchant')
axes[1, 2].set_ylabel('Transactions (%)')
for position, value in enumerate(merchant_share):
    axes[1, 2].text(position, value, f'{value:.1f}%', ha='center', va='bottom')

fig.suptitle('General Customer-History Feature Distributions', fontsize=16)
plt.savefig(
    FIGURE_DIR / "figure-3.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


customer_history_general_summary = pd.Series({
    'median_prior_customer_transactions': df['customer_prior_txn_count'].median(),
    'median_minutes_since_previous_transaction': df.loc[
        df['minutes_since_previous_txn'].ge(0), 'minutes_since_previous_txn'
    ].median(),
    'median_amount_to_prior_mean': df.loc[
        df['no_prior_amount_history'].eq(0), 'amount_to_prior_mean'
    ].median(),
    'first_observed_transaction_share': df['is_first_observed_txn'].mean(),
    'new_merchant_transaction_share': df['is_new_merchant_for_customer'].mean(),
})
display(customer_history_general_summary.to_frame('value'))

### EDA: Customer History w/ Fraud

In [ ]:
def quantile_fraud_summary(data, feature, valid_mask=None, quantiles=10):
    subset = data[[feature, 'is_fraud']].copy()
    if valid_mask is not None:
        subset = subset.loc[valid_mask]
    subset = subset.dropna()
    subset['_bin'] = pd.qcut(subset[feature], q=quantiles, duplicates='drop')
    return subset.groupby('_bin', observed=True)['is_fraud'].agg(
        transactions='size', fraud_cases='sum', fraud_rate='mean'
    )


def binary_fraud_summary(data, feature):
    return data.groupby(feature, observed=True)['is_fraud'].agg(
        transactions='size', fraud_cases='sum', fraud_rate='mean'
    ).reindex([0, 1])


history_fraud_summaries = {
    'Prior customer transaction count': quantile_fraud_summary(
        df, 'customer_prior_txn_count'
    ),
    'Minutes since previous transaction': quantile_fraud_summary(
        df, 'minutes_since_previous_txn', df['minutes_since_previous_txn'].ge(0)
    ),
    'Amount relative to prior average': quantile_fraud_summary(
        df, 'amount_to_prior_mean', df['no_prior_amount_history'].eq(0)
    ),
    'Prior customer–merchant count': quantile_fraud_summary(
        df, 'customer_merchant_prior_count'
    ),
    'First observed transaction': binary_fraud_summary(
        df, 'is_first_observed_txn'
    ),
    'New merchant for customer': binary_fraud_summary(
        df, 'is_new_merchant_for_customer'
    ),
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)

for ax, (title, summary) in zip(axes.ravel(), history_fraud_summaries.items()):
    positions = np.arange(len(summary))
    ax.bar(positions, summary['fraud_rate'].mul(100), color='#E15759', alpha=0.82)
    ax.set_title(title)
    ax.set_ylabel('Fraud rate (%)', color='#B23A3A')
    ax.tick_params(axis='y', labelcolor='#B23A3A')

    if len(summary) == 2:
        labels = ['No', 'Yes']
    else:
        labels = [f'Q{number}' for number in range(1, len(summary) + 1)]
    ax.set_xticks(positions)
    ax.set_xticklabels(labels)
    ax.set_xlabel('Low → high quantile' if len(summary) > 2 else '')

    cases_axis = ax.twinx()
    cases_axis.plot(positions, summary['fraud_cases'], color='#1F4E79', marker='o')
    cases_axis.set_ylabel('Fraud cases', color='#1F4E79')
    cases_axis.tick_params(axis='y', labelcolor='#1F4E79')

    rate_proxy = plt.Line2D([0], [0], color='#E15759', linewidth=8, alpha=0.82)
    cases_proxy = plt.Line2D([0], [0], color='#1F4E79', marker='o')
    ax.legend([rate_proxy, cases_proxy], ['Fraud rate', 'Fraud cases'],
              loc='upper left', fontsize=8)

fig.suptitle('Customer-History Features: Fraud Rate and Fraud-Case Volume', fontsize=16)
plt.savefig(
    FIGURE_DIR / "figure-4.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

history_flag_fraud_summary = pd.DataFrame({
    'fraud_rate_without_flag': [
        history_fraud_summaries['First observed transaction'].loc[0, 'fraud_rate'],
        history_fraud_summaries['New merchant for customer'].loc[0, 'fraud_rate'],
    ],
    'fraud_rate_with_flag': [
        history_fraud_summaries['First observed transaction'].loc[1, 'fraud_rate'],
        history_fraud_summaries['New merchant for customer'].loc[1, 'fraud_rate'],
    ],
}, index=['First observed transaction', 'New merchant for customer'])
history_flag_fraud_summary['risk_ratio'] = (
    history_flag_fraud_summary['fraud_rate_with_flag']
    / history_flag_fraud_summary['fraud_rate_without_flag']
)
display(history_flag_fraud_summary)

#### Interpretation and behavioral-rule relevance

- **Prior customer transactions:** Fraud rate declines as prior observed activity increases, from roughly 1.5% in the lowest-history group to roughly 0.3% in the highest-history group. A longer history provides a more stable picture of normal behavior. Limited history creates uncertainty because fewer past observations are available for comparison. This is an uncertainty signal, not proof that a new customer is fraudulent.
- **Minutes since the previous transaction:** The U-shaped pattern is consistent with two different risks. Very short gaps can represent a rapid fraud burst, card testing, or an attempt to complete several purchases before the customer or issuer reacts. Very long gaps can represent use of a dormant or rarely used card, a stale behavioral baseline, or account reactivation after compromise. Both patterns also have legitimate explanations—shopping sessions at the short end and returning customers at the long end—so interval extremes should be combined with model score and other anomalies.
- **Amount relative to the prior average:** Fraud rate is comparatively stable across the first nine groups and rises sharply in the highest group. A transaction substantially larger than the customer's prior average is therefore more useful than raw amount alone. Fraudsters may attempt a large purchase before the card is blocked, but legitimate major purchases remain possible.
- **Customer–merchant history:** Fraud is more common when the customer has little or no history with the merchant. A previously used merchant provides evidence of an established relationship, while a new merchant is more uncertain.
- **First observed transaction:** The approximately 7–8% observed fraud rate is much higher than for profiles with history, but the group is small. The flag represents missing behavioral evidence and must not independently trigger a decline.
- **New merchant:** New-merchant transactions have a higher observed fraud rate than transactions at previously visited merchants. This is strongest when combined with an amount spike, risky time, or high-risk category.



### Class Imbalance

In [ ]:
class_summary = df['is_fraud'].value_counts().sort_index().rename_axis('is_fraud').to_frame('count')
class_summary['share'] = class_summary['count'] / class_summary['count'].sum()
display(class_summary)

ax = class_summary['count'].plot(kind='bar', color=['#4C78A8', '#E45756'], figsize=(7, 4))
ax.set_title('Class Imbalance: Non-fraud vs Fraud')
ax.set_xlabel('is_fraud')
ax.set_ylabel('Transactions')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

### EDA: Shopping Category w/ Fraud

In [ ]:
category_analysis = (
    df.groupby('category', observed=True)
      .agg(transactions=('is_fraud', 'size'), fraud_cases=('is_fraud', 'sum'), fraud_rate=('is_fraud', 'mean'))
      .sort_values('fraud_rate', ascending=False)
)
display(category_analysis)

ax = category_analysis['fraud_rate'].sort_values().plot(
    kind='barh', color='#E45756', figsize=(8, 6)
)
ax.set_title('Observed Fraud Rate by Merchant Category')
ax.set_xlabel('Fraud rate')
ax.set_ylabel('Category')
plt.tight_layout()
plt.show()

### Category Level: Continue
- `shopping_net` has the highest observed fraud rate, while `grocery_pos` has the most fraud cases. This difference occurs because case volume depends on both the category’s risk and its total number of transactions.
- Online categories (`shopping_net` and `misc_net`) combine high fraud rates with much larger median fraudulent amounts than legitimate amounts. This is consistent with the dataset assigning a distinctive high-value fraud pattern to remote shopping, but it does not establish a real-world causal mechanism.
- `grocery_pos` combines high transaction volume with a high fraud rate and becomes substantially riskier at night. This explains why it produces the largest fraud workload even though its overall rate is below `shopping_net`.
- `gas_transport` and `grocery_net` also show stronger night-time fraud rates than their day/evening rates, suggesting an interaction between category and transaction time.
- Category should therefore remain a model feature and may interact with amount and time. Production rules should not decline solely because of category; they should combine category with model score, amount, customer history, merchant familiarity, and timing.

In [ ]:
category_risk = (
    df.groupby('category', observed=True)
      .agg(
          transactions=('is_fraud', 'size'),
          fraud_cases=('is_fraud', 'sum'),
          fraud_rate=('is_fraud', 'mean'),
          total_transaction_amount=('amt', 'sum'),
      )
)

category_amount = (
    df.groupby(['category', 'is_fraud'], observed=True)['amt']
      .median()
      .unstack('is_fraud')
      .rename(columns={0: 'median_nonfraud_amount', 1: 'median_fraud_amount'})
)

category_time = (
    df.groupby(['category', 'is_night'], observed=True)['is_fraud']
      .agg(transactions='size', fraud_rate='mean')
      .reset_index()
)
day_rate = (
    category_time.loc[category_time['is_night'].eq(0)]
      .set_index('category')['fraud_rate']
      .rename('day_evening_fraud_rate')
)
night_rate = (
    category_time.loc[category_time['is_night'].eq(1)]
      .set_index('category')['fraud_rate']
      .rename('night_fraud_rate')
)

fraud_amount = (
    df.loc[df['is_fraud'].eq(1)]
      .groupby('category', observed=True)['amt']
      .sum()
      .rename('fraud_dollars')
)

category_risk = (
    category_risk.join(category_amount)
                 .join(day_rate)
                 .join(night_rate)
                 .join(fraud_amount)
)
category_risk['fraud_amount_multiplier'] = (
    category_risk['median_fraud_amount']
    / category_risk['median_nonfraud_amount'].replace(0, np.nan)
)

rate_order = category_risk.sort_values('fraud_rate').index
case_order = category_risk.sort_values('fraud_cases').index
amount_order = category_risk.sort_values('median_fraud_amount').index

fig, axes = plt.subplots(2, 2, figsize=(17, 12), constrained_layout=True)

category_risk.loc[rate_order, 'fraud_rate'].mul(100).plot.barh(
    ax=axes[0, 0], color='#E15759'
)
axes[0, 0].axvline(
    df['is_fraud'].mean() * 100, color='gray', linestyle='--', label='Overall fraud rate'
)
axes[0, 0].set_title('Fraud Rate by Transaction Category')
axes[0, 0].set_xlabel('Fraud rate (%)')
axes[0, 0].set_ylabel('Category')
axes[0, 0].legend()

category_risk.loc[case_order, 'fraud_cases'].plot.barh(
    ax=axes[0, 1], color='#4C78A8'
)
axes[0, 1].set_title('Total Fraud Cases by Transaction Category')
axes[0, 1].set_xlabel('Fraud cases')
axes[0, 1].set_ylabel('Category')
axes[0, 1].xaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:,.0f}'))

amount_plot = category_risk.loc[amount_order, [
    'median_nonfraud_amount', 'median_fraud_amount'
]]
amount_plot.plot.barh(ax=axes[1, 0], color=['#59A14F', '#F28E2B'])
axes[1, 0].set_title('Median Transaction Amount: Fraud vs Non-fraud')
axes[1, 0].set_xlabel('Median transaction amount ($, log scale)')
axes[1, 0].set_ylabel('Category')
axes[1, 0].set_xscale('log')
axes[1, 0].legend(['Non-fraud', 'Fraud'])

top_categories = category_risk.nlargest(8, 'fraud_rate').index
time_plot = category_risk.loc[top_categories, [
    'day_evening_fraud_rate', 'night_fraud_rate'
]].mul(100).sort_values('day_evening_fraud_rate')
time_plot.plot.barh(ax=axes[1, 1], color=['#76B7B2', '#B07AA1'])
axes[1, 1].set_title('Top-risk Categories: Fraud Rate by Time Period')
axes[1, 1].set_xlabel('Fraud rate (%)')
axes[1, 1].set_ylabel('Category')
axes[1, 1].legend(['06:00–23:59', '00:00–05:59'])

fig.suptitle('Category Fraud Risk, Volume, Amount, and Time Links', fontsize=16)
plt.savefig(
    FIGURE_DIR / "figure-5.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

category_risk_display = (
    category_risk.sort_values(['fraud_rate', 'fraud_cases'], ascending=False)
    [[
        'transactions', 'fraud_cases', 'fraud_rate', 'fraud_dollars',
        'median_nonfraud_amount', 'median_fraud_amount',
        'fraud_amount_multiplier', 'day_evening_fraud_rate', 'night_fraud_rate',
    ]]
)
display(category_risk_display.style.format({
    'transactions': '{:,.0f}', 'fraud_cases': '{:,.0f}',
    'fraud_rate': '{:.2%}', 'fraud_dollars': '${:,.2f}',
    'median_nonfraud_amount': '${:,.2f}', 'median_fraud_amount': '${:,.2f}',
    'fraud_amount_multiplier': '{:.1f}×',
    'day_evening_fraud_rate': '{:.2%}', 'night_fraud_rate': '{:.2%}',
}))

highest_rate_category = category_risk['fraud_rate'].idxmax()
highest_case_category = category_risk['fraud_cases'].idxmax()
highest_fraud_amount_category = category_risk['fraud_dollars'].idxmax()

category_key_findings = pd.Series({
    'category_with_highest_fraud_rate': highest_rate_category,
    'highest_category_fraud_rate': category_risk.loc[highest_rate_category, 'fraud_rate'],
    'category_with_most_fraud_cases': highest_case_category,
    'most_category_fraud_cases': int(category_risk.loc[highest_case_category, 'fraud_cases']),
    'category_with_most_fraud_dollars': highest_fraud_amount_category,
    'most_category_fraud_dollars': category_risk.loc[highest_fraud_amount_category, 'fraud_dollars'],
})
display(category_key_findings.to_frame('value'))

In [ ]:
display(df.groupby('is_fraud')['amt'].describe())

df['amt_log1p'] = np.log1p(df['amt']).astype('float32')

ax = df.boxplot(
    column='amt_log1p',
    by='is_fraud',
    figsize=(7, 4),
    grid=False,
)
ax.set_title('Transaction Amount Distribution by Fraud Status')
ax.set_xlabel('Transaction class (0 = non-fraud, 1 = fraud)')
ax.set_ylabel('Log-transformed amount: log(1 + amount)')
plt.suptitle('')
plt.tight_layout()
plt.show()

### Feature by Feature Association
For continuous variables, we will use absolute correlation and Standardized Mean Difference. For categorical variables, we will use the Cramer's V. For all variables, we will use its relation corresponds to `is_fraud`

From the result, we know that `amt` is one of the key determinator, meaning fraud transactions are much larger than normal ones. Other, such as `amount_to_prior_mean` can also be useful, since spending much more than the customer's normal behavior is suspicious.

`zip` is the strongest categorical predictor, fraud is concentrated in certain ZIP codes.


In [ ]:
# Section 6 preflight: rebuild engineered dependencies when this section is
# executed after a kernel restart or without manually running every feature cell.
required_source_columns = {
    'trans_date_trans_time', 'dob', 'amt', 'lat', 'long', 'merch_lat',
    'merch_long', 'first', 'last', 'street', 'merchant', 'is_fraud',
}
missing_source_columns = sorted(required_source_columns.difference(df.columns))
if missing_source_columns:
    raise KeyError(
        'Section 6 cannot run because source columns are missing: '
        f'{missing_source_columns}. Run Sections 1–3 first.'
    )

# Time and amount features.
if not pd.api.types.is_datetime64_any_dtype(df['trans_date_trans_time']):
    df['trans_date_trans_time'] = pd.to_datetime(
        df['trans_date_trans_time'], format='mixed', dayfirst=True, errors='coerce'
    )
if not pd.api.types.is_datetime64_any_dtype(df['dob']):
    df['dob'] = pd.to_datetime(
        df['dob'], format='mixed', dayfirst=True, errors='coerce'
    )

transaction_time = df['trans_date_trans_time']
df['amt_log1p'] = np.log1p(df['amt']).astype('float32')
df['trans_hour'] = transaction_time.dt.hour.astype('int8')
df['trans_day_of_week'] = transaction_time.dt.dayofweek.astype('int8')
df['trans_month'] = transaction_time.dt.month.astype('int8')
df['is_weekend'] = transaction_time.dt.dayofweek.ge(5).astype('int8')
df['is_night'] = transaction_time.dt.hour.between(0, 5).astype('int8')
df['age_at_transaction'] = (
    (transaction_time - df['dob']).dt.days / 365.2425
).clip(lower=0).astype('float32')
df['hour_sin'] = np.sin(2 * np.pi * df['trans_hour'] / 24).astype('float32')
df['hour_cos'] = np.cos(2 * np.pi * df['trans_hour'] / 24).astype('float32')
df['day_sin'] = np.sin(2 * np.pi * df['trans_day_of_week'] / 7).astype('float32')
df['day_cos'] = np.cos(2 * np.pi * df['trans_day_of_week'] / 7).astype('float32')

# Customer-to-merchant geographic features.
df['merchant_lat_delta'] = (df['merch_lat'] - df['lat']).astype('float32')
df['merchant_long_delta'] = (df['merch_long'] - df['long']).astype('float32')

if 'haversine_km' not in globals():
    def haversine_km(lat1, lon1, lat2, lon2):
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        delta_lat = lat2 - lat1
        delta_lon = lon2 - lon1
        a = (
            np.sin(delta_lat / 2) ** 2
            + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2) ** 2
        )
        return 2 * 6371.0088 * np.arcsin(np.sqrt(a))

df['merchant_distance_km'] = haversine_km(
    df['lat'].to_numpy(), df['long'].to_numpy(),
    df['merch_lat'].to_numpy(), df['merch_long'].to_numpy(),
).astype('float32')
bearing_radians = np.arctan2(
    np.radians(df['merchant_long_delta']),
    np.radians(df['merchant_lat_delta']),
)
df['merchant_direction_sin'] = np.sin(bearing_radians).astype('float32')
df['merchant_direction_cos'] = np.cos(bearing_radians).astype('float32')

# Rebuild causal customer-history features only if they are absent.
history_features = {
    'customer_prior_txn_count', 'minutes_since_previous_txn',
    'is_first_observed_txn', 'amount_to_prior_mean',
    'no_prior_amount_history', 'customer_merchant_prior_count',
    'is_new_merchant_for_customer',
}
if not history_features.issubset(df.columns):
    customer_profiles = ['first', 'last', 'dob', 'street']
    df['customer_key'] = pd.util.hash_pandas_object(
        df[customer_profiles].astype('string'), index=False
    ).astype('uint64').astype('string')

    original_order = pd.Series(np.arange(len(df)), index=df.index, name='_original_order')
    df['_original_order'] = original_order
    df = df.sort_values(['customer_key', 'trans_date_trans_time'], kind='stable').copy()
    customer_group = df.groupby('customer_key', sort=False)

    df['customer_prior_txn_count'] = customer_group.cumcount().astype('int32')
    df['minutes_since_previous_txn'] = (
        customer_group['trans_date_trans_time'].diff().dt.total_seconds().div(60)
    ).astype('float32')
    df['is_first_observed_txn'] = df['minutes_since_previous_txn'].isna().astype('int8')

    prior_amount_sum = customer_group['amt'].cumsum() - df['amt']
    prior_amount_mean = prior_amount_sum.div(
        df['customer_prior_txn_count'].replace(0, np.nan)
    )
    df['amount_to_prior_mean'] = (
        df['amt'] / prior_amount_mean.replace(0, np.nan)
    ).astype('float32')
    df['no_prior_amount_history'] = prior_amount_mean.isna().astype('int8')
    df['customer_merchant_prior_count'] = (
        df.groupby(['customer_key', 'merchant'], sort=False).cumcount().astype('int32')
    )
    df['is_new_merchant_for_customer'] = (
        df['customer_merchant_prior_count'].eq(0).astype('int8')
    )
    df['minutes_since_previous_txn'] = df['minutes_since_previous_txn'].fillna(-1)
    df['amount_to_prior_mean'] = (
        df['amount_to_prior_mean'].replace([np.inf, -np.inf], np.nan).fillna(1)
    )
    df = df.sort_values('_original_order', kind='stable').drop(columns='_original_order')

required_engineered_features = {
    'amt_log1p', 'trans_hour', 'trans_day_of_week', 'trans_month',
    'age_at_transaction', 'merchant_lat_delta', 'merchant_long_delta',
    'merchant_distance_km', 'merchant_direction_sin', 'merchant_direction_cos',
} | history_features
missing_engineered_features = sorted(required_engineered_features.difference(df.columns))
assert not missing_engineered_features, (
    f'Engineered features are still missing: {missing_engineered_features}'
)
print('Section 6 feature dependencies are ready.')

In [ ]:
def standardized_mean_difference(data, feature, target='is_fraud'):
    group_0 = data.loc[data[target].eq(0), feature].dropna()
    group_1 = data.loc[data[target].eq(1), feature].dropna()
    pooled_variance = (
        ((len(group_0) - 1) * group_0.var() + (len(group_1) - 1) * group_1.var())
        / (len(group_0) + len(group_1) - 2)
    )
    return (group_1.mean() - group_0.mean()) / np.sqrt(pooled_variance)


def corrected_cramers_v(data, feature, target='is_fraud'):
    table = pd.crosstab(data[feature], data[target]).to_numpy()
    n = table.sum()
    expected = table.sum(axis=1)[:, None] * table.sum(axis=0)[None, :] / n
    chi_squared = np.divide(
        (table - expected) ** 2, expected,
        out=np.zeros_like(expected, dtype='float64'), where=expected != 0,
    ).sum()
    rows, columns = table.shape
    phi_squared = chi_squared / n
    corrected_phi = max(0, phi_squared - ((columns - 1) * (rows - 1)) / (n - 1))
    corrected_rows = rows - ((rows - 1) ** 2) / (n - 1)
    corrected_columns = columns - ((columns - 1) ** 2) / (n - 1)
    denominator = min(corrected_rows - 1, corrected_columns - 1)
    return np.sqrt(corrected_phi / denominator) if denominator > 0 else 0.0


numeric_features = [
    'amt', 'amt_log1p', 'city_pop', 'lat', 'long', 'merch_lat', 'merch_long',
    'trans_hour', 'trans_day_of_week', 'trans_month', 'age_at_transaction',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'merchant_lat_delta', 'merchant_long_delta', 'merchant_distance_km',
    'merchant_direction_sin', 'merchant_direction_cos',
    'customer_prior_txn_count', 'minutes_since_previous_txn',
    'amount_to_prior_mean', 'customer_merchant_prior_count',
]
categorical_features = [
    'merchant', 'category', 'gender', 'city', 'state', 'zip', 'job',
    'is_weekend', 'is_night', 'is_first_observed_txn',
    'no_prior_amount_history', 'is_new_merchant_for_customer',
]

numeric_association = pd.DataFrame([
    {
        'feature': feature,
        'absolute_correlation': abs(df[feature].corr(df['is_fraud'])),
        'standardized_mean_difference': standardized_mean_difference(df, feature),
        'unique_values': df[feature].nunique(),
    }
    for feature in numeric_features
]).sort_values('absolute_correlation', ascending=False)

categorical_association = pd.DataFrame([
    {
        'feature': feature,
        'corrected_cramers_v': corrected_cramers_v(df, feature),
        'unique_values': df[feature].nunique(),
    }
    for feature in categorical_features
]).sort_values('corrected_cramers_v', ascending=False)

display(numeric_association)
display(categorical_association)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7), constrained_layout=True)

numeric_association.sort_values('absolute_correlation').plot.barh(
    x='feature', y='absolute_correlation', ax=axes[0], legend=False, color='#4C78A8'
)
axes[0].set_title('Numeric Features: Absolute Correlation with Fraud')
axes[0].set_xlabel('Absolute point-biserial correlation')
axes[0].set_ylabel('')

categorical_association.sort_values('corrected_cramers_v').plot.barh(
    x='feature', y='corrected_cramers_v', ax=axes[1], legend=False, color='#F58518'
)
axes[1].set_title("Categorical Features: Bias-corrected Cramér's V")
axes[1].set_xlabel("Cramér's V effect size")
axes[1].set_ylabel('')
plt.show()

**What this block does:** Converts the association tables into two horizontal bar charts. The charts make it easier to compare which numeric and categorical features have the strongest univariate relationship with fraud.

### Numeric feature shapes
For this part, we will divide the feature into 10 equal size quantiles, then we will compute the fraud rate within each quantile.
`amt` reveals that very large transactions are much riskier. `amount_to_prior_mean` means transaction much larger than a customer's historical average have dramatically higher fraud rates

In [ ]:
def plot_numeric_fraud_rates(data, features, columns=3, quantiles=10):
    rows = int(np.ceil(len(features) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(5 * columns, 3.6 * rows))
    axes = np.atleast_1d(axes).ravel()
    overall_rate = data['is_fraud'].mean() * 100

    for ax, feature in zip(axes, features):
        valid = data[[feature, 'is_fraud']].dropna()
        bins = pd.qcut(valid[feature], q=quantiles, duplicates='drop')
        rates = valid.groupby(bins, observed=True)['is_fraud'].mean().mul(100)
        ax.plot(range(len(rates)), rates.values, marker='o', color='#E45756')
        ax.axhline(overall_rate, color='gray', linestyle='--', linewidth=1)
        ax.set_title(feature)
        ax.set_xlabel('Low → high quantile')
        ax.set_ylabel('Fraud rate (%)')

    for ax in axes[len(features):]:
        ax.set_visible(False)
    fig.suptitle('Fraud Rate Across Numeric Feature Quantiles', fontsize=15, y=1.01)
    plt.tight_layout()
    plt.show()


plot_numeric_fraud_rates(df, numeric_features)

### Categorical feature shapes

For low-cardinality features, every level is plotted. For high-cardinality features, the chart requires a minimum number of transactions and shows the levels with the largest absolute difference from the overall fraud rate. This avoids charts dominated by tiny groups. High-cardinality target rates must not be reused directly as model features without training-only, out-of-fold encoding.

In [ ]:
def plot_all_categorical_fraud_rates(
        data,
        features,
        columns=3,
        top_n=15,
        min_count=500,
):
    rows = int(np.ceil(len(features) / columns))
    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(5 * columns, 3.8 * rows),
        constrained_layout=True,
    )

    axes = np.atleast_1d(axes).ravel()
    overall_rate = data["is_fraud"].mean()

    for ax, feature in zip(axes, features):
        summary = (
            data.groupby(feature, observed=True)["is_fraud"]
            .agg(transactions="size", fraud_rate="mean")
            .dropna()
        )

        if len(summary) > top_n:
            summary = summary.loc[summary["transactions"] >= min_count].copy()
            summary["absolute_lift"] = (
                    summary["fraud_rate"] - overall_rate
            ).abs()
            summary = summary.nlargest(top_n, "absolute_lift")

        summary = summary.sort_values("fraud_rate")

        ax.barh(
            summary.index.astype(str),
            summary["fraud_rate"] * 100,
            color="#4C78A8",
        )

        ax.axvline(
            overall_rate * 100,
            color="#E45756",
            linestyle="--",
            linewidth=1,
        )

        ax.set_title(feature, fontsize=10)
        ax.set_xlabel("Fraud rate (%)", fontsize=8)
        ax.tick_params(axis="both", labelsize=8)

    # Hide unused axes
    for ax in axes[len(features):]:
        ax.set_visible(False)

    fig.suptitle(
        "Observed Fraud Rate Across Categorical Features",
        fontsize=15,
    )

    plt.show()


plot_all_categorical_fraud_rates(
    df,
    categorical_features,
    columns=3,  # 3x4 layout
)

### Complete column audit

The table below ensures every original or engineered column receives an explicit modeling decision. Strong univariate association is useful for screening, but final importance must be measured with a time-aware validation set and model-based permutation importance.

In [ ]:
excluded_identifiers = {
    'cc_num', 'first', 'last', 'street', 'dob', 'trans_num', 'customer_key'
}
raw_time_fields = {'trans_date_trans_time'}

audit_rows = []
for feature in df.columns:
    if feature == 'is_fraud':
        role, measure, effect = 'target', None, None
    elif feature in excluded_identifiers:
        role, measure, effect = 'exclude: identifier/PII', None, None
    elif feature in raw_time_fields:
        role, measure, effect = 'retain for chronological splitting', None, None
    elif feature in numeric_features:
        row = numeric_association.set_index('feature').loc[feature]
        role, measure, effect = 'candidate numeric predictor', 'absolute correlation', row['absolute_correlation']
    elif feature in categorical_features:
        row = categorical_association.set_index('feature').loc[feature]
        role, measure, effect = 'candidate categorical predictor', "corrected Cramér's V", row['corrected_cramers_v']
    elif feature in ['hour_sin', 'hour_cos', 'day_sin', 'day_cos',
                     'merchant_direction_sin', 'merchant_direction_cos']:
        role, measure, effect = 'candidate cyclic predictor', 'evaluate with model validation', None
    else:
        role, measure, effect = 'review', None, None

    audit_rows.append({
        'feature': feature,
        'dtype': str(df[feature].dtype),
        'unique_values': df[feature].nunique(dropna=False),
        'missing_values': int(df[feature].isna().sum()),
        'role': role,
        'association_measure': measure,
        'univariate_effect': effect,
    })

column_audit = pd.DataFrame(audit_rows)
display(column_audit)

## 7. Create and save the modeling table

In [ ]:
drop_from_model = [
    'cc_num', 'first', 'last', 'street', 'dob', 'trans_num', 'customer_key'
]

df_model = (
    df.drop(columns=drop_from_model)
      .sort_values('trans_date_trans_time', kind='stable')
      .reset_index(drop=True)
)

assert len(df_model) == len(df), 'Unexpected row loss during preprocessing'
assert not df_model.isna().any().any(), 'Processed data contains missing values'
assert set(df_model['is_fraud'].unique()) == {0, 1}
assert df_model['merchant_distance_km'].ge(0).all()
assert df_model['age_at_transaction'].between(0, 120).all()

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df_model.to_csv(PROCESSED_PATH, index=False, date_format='%Y-%m-%d %H:%M:%S')

print(f'All validation checks passed. Saved {len(df_model):,} rows and {df_model.shape[1]} columns to:')
print(PROCESSED_PATH)